# Add FGCM Photometric Calibration Information to Light Curves

This notebook enriches the light-curve files produced by
`05_AddLSSTCalibParam.ipynb` (PhotoCalib-based calibration parameters) with
**complementary calibration information from FGCM** (Forward Global Calibration
Method), retrieved from the Butler.

## Rationale

Notebook `05_AddLSSTCalibParam.ipynb` extracts per-visit, per-detector
calibration parameters (`calib_mean`, `calib_local`, `zeropoint`) directly
from the `PhotoCalib` object embedded in each Processed Visit Image (PVI).

FGCM is a **global, forward-modelled** photometric calibration that is fit
jointly over the whole survey (all visits, all stars) and provides:

- A **global FGCM zero-point** per visit (and usually per detector), which is
  an independent calibration estimate that can be compared to the PhotoCalib
  zero-point already stored in the light curves.
- **Atmospheric parameters** per visit (PWV, ozone, aerosol optical depth
  `tau`, Angstrom exponent `alpha`, ...), which are useful for studying the
  physical origin of calibration variations (see the companion AuxTel/
  Spectractor PWV analysis and the DCR/dipole studies).

## Important notes

- **Not every visit has an FGCM solution.** Some visits are rejected by the
  FGCM fit (e.g. bad conditions, insufficient standard stars) or simply fall
  outside the FGCM run's input visit list. Missing FGCM information is
  therefore expected and is flagged rather than treated as an error.
- The exact FGCM dataset type names depend on the **fit cycle number**
  (`fgcm_Cycle<N>_AtmosphereParameters`, `fgcm_Cycle<N>_Zeropoints`, ...),
  which changes between DRP reprocessings. This notebook **auto-discovers**
  the available FGCM dataset types in the registry and picks the highest
  cycle number found, with manual override variables provided in the
  configuration cell in case auto-discovery picks the wrong one.
- FGCM zero-points are usually tabulated **per (visit, detector)**. The match
  against the light curves therefore uses the *reliable* `detector_id` column
  added by notebook `05_AddLSSTCalibParam.ipynb` (not the unreliable `detector`
  column from the raw source table).

## What is added per data point

| New column | Source | Meaning |
|---|---|---|
| `fgcm_zeropoint` | FGCM Zeropoints dataset | FGCM-fitted photometric zero-point |
| `fgcm_zeropoint_err` | FGCM Zeropoints dataset | Uncertainty on the FGCM zero-point (if available) |
| `fgcm_pwv` | FGCM AtmosphereParameters dataset | Precipitable water vapour (mm) |
| `fgcm_o3` | FGCM AtmosphereParameters dataset | Ozone column |
| `fgcm_tau` | FGCM AtmosphereParameters dataset | Aerosol optical depth |
| `fgcm_alpha` | FGCM AtmosphereParameters dataset | Aerosol Angstrom exponent |
| `fgcm_lntau` | FGCM AtmosphereParameters dataset | log(tau), if provided instead of tau |
| `fgcm_pmb` | FGCM AtmosphereParameters dataset | Barometric pressure |
| `fgcm_atm_flag` | derived | `True` if atmospheric parameters were found for this visit |
| `fgcm_zpt_flag` | derived | `True` if an FGCM zero-point was found for this (visit, detector) |
| `fgcm_flag` | derived | `True` if `fgcm_atm_flag` OR `fgcm_zpt_flag` |

Only the atmospheric-parameter columns that actually exist in the FGCM
dataset for this collection are added (auto-detected — see configuration).

## Input
- `data_AddCalib_05_out/all_stars_lightcurves_calib.csv` (global)
- `data_AddCalib_05_out/per_star/*_lc_calib.csv` (per-star)

## Output
- `data_AddFGCM_08_out/all_stars_lightcurves_fgcm.csv` + `.parquet`
- `data_AddFGCM_08_out/per_star/*_lc_fgcm.csv` + `.parquet`

---
- **Author:** Sylvie Dagoret-Campagne
- **Affiliation:** IJCLab/IN2P3/CNRS, Université Paris-Saclay
- **Created:** 2026-07-01
- **Last update:** 2026-07-01


## 1. Imports

In [ ]:
import gc
import logging
import os
import re
import sys

import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

from astropy.time import Time

from lsst.daf.butler import Butler

from libExtractLightcurves import safe_name, find_col

## 2. Logging setup

In [ ]:
log = logging.getLogger()
log.setLevel(logging.INFO)

if not log.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setLevel(logging.INFO)
    formatter = logging.Formatter("%(asctime)s - %(levelname)s - %(message)s")
    handler.setFormatter(formatter)
    log.addHandler(handler)

log.info("Logging initialised.")

## 3. Configuration

In [ ]:
# ── Notebook tag ─────────────────────────────────────────────────────────────
NB_TAG = "AddFGCM_08"

# ── Input: light curves already enriched with PhotoCalib info (output of notebook 05) ─
DIR_LC_IN = "./data_AddCalib_05_out"
DIR_LC_PER_STAR_IN = os.path.join(DIR_LC_IN, "per_star")
GLOBAL_LC_FILE = "all_stars_lightcurves_calib.csv"

# ── Output ────────────────────────────────────────────────────────────────────
DIR_DATA = f"./data_{NB_TAG}_out"
DIR_DATA_PER_STAR = os.path.join(DIR_DATA, "per_star")
DIR_FIGS = f"./figs_{NB_TAG}"

for d in [DIR_DATA, DIR_DATA_PER_STAR, DIR_FIGS]:
    os.makedirs(d, exist_ok=True)
    log.info("Directory ready: %s", d)

# ── Butler configuration (same repo/collection as notebook 05) ───────────────
repo = "dp2_prep"
collection = [
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage1",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage2",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage3",
    "LSSTCam/runs/DRP/DP2/v30_0_0/DM-53881/stage4",
]

# ── FGCM dataset type auto-discovery ──────────────────────────────────────────
# FGCM dataset type names embed a fit "cycle" number that changes between DRP
# reprocessings (e.g. fgcm_Cycle5_AtmosphereParameters). Set the override
# variables below to force a specific dataset type name if auto-discovery
# picks the wrong one (print the discovered candidates below and inspect).
FGCM_ATM_DATASET_OVERRIDE = None  # e.g. "fgcm_Cycle5_AtmosphereParameters"
FGCM_ZPT_DATASET_OVERRIDE = None  # e.g. "fgcm_Cycle5_Zeropoints"

# Candidate atmospheric parameter column names to look for in the FGCM
# AtmosphereParameters table. Only the ones actually present are kept.
ATM_PARAM_CANDIDATES = ["pwv", "o3", "tau", "alpha", "lnTau", "pmb"]

# Candidate column names for the FGCM zero-point and its uncertainty.
ZP_COL_CANDIDATES = ["fgcmZpt", "fgcm_zpt", "zpt", "zeropoint", "zp"]
ZP_ERR_COL_CANDIDATES = ["fgcmZptErr", "fgcm_zpt_err", "zptErr", "zeropointErr", "zpErr"]

# Candidate column names for the detector id in the FGCM zero-point table.
DET_COL_CANDIDATES = ["detector", "ccd", "detectorId", "detector_id"]

# Candidate column names for the visit id (should normally just be "visit").
VISIT_COL_CANDIDATES = ["visit", "visitId", "visit_id"]

log.info("Configuration done.")

## 4. Initialise the Butler and discover FGCM dataset types

In [ ]:
butler = Butler(repo, collections=collection)
registry = butler.registry
log.info("Butler initialised | repo: %s", repo)

In [ ]:
def discover_fgcm_datasets(registry, name_pattern: str) -> list[str]:
    """Return dataset type names in the registry matching *name_pattern* (case-insensitive regex)."""
    return sorted(
        d.name for d in registry.queryDatasetTypes() if re.search(name_pattern, d.name, re.IGNORECASE)
    )


def pick_latest_cycle(names: list[str]) -> str | None:
    """Among candidate dataset type names, pick the one with the highest 'Cycle<N>' number.

    Falls back to the first candidate (alphabetical) if no cycle number is found.
    """
    if not names:
        return None

    def cycle_num(n: str) -> int:
        m = re.search(r"Cycle(\d+)", n, re.IGNORECASE)
        return int(m.group(1)) if m else -1

    return sorted(names, key=cycle_num)[-1]


atm_candidates = discover_fgcm_datasets(registry, r"fgcm.*AtmosphereParameters")
zpt_candidates = discover_fgcm_datasets(registry, r"fgcm.*Zeropoints?")

log.info("FGCM AtmosphereParameters candidates found: %s", atm_candidates)
log.info("FGCM Zeropoints candidates found: %s", zpt_candidates)

ATM_DATASET = FGCM_ATM_DATASET_OVERRIDE or pick_latest_cycle(atm_candidates)
ZPT_DATASET = FGCM_ZPT_DATASET_OVERRIDE or pick_latest_cycle(zpt_candidates)

log.info("Selected FGCM AtmosphereParameters dataset type: %s", ATM_DATASET)
log.info("Selected FGCM Zeropoints dataset type: %s", ZPT_DATASET)

if ATM_DATASET is None and ZPT_DATASET is None:
    raise RuntimeError(
        "No FGCM AtmosphereParameters nor Zeropoints dataset type found in this "
        "Butler collection. Check the 'collection' configuration."
    )

## 5. Load the FGCM tables from the Butler

In [ ]:
def load_fgcm_table(butler: Butler, dataset_name: str | None) -> pd.DataFrame | None:
    """Load an FGCM dataset (a lsst.afw.table / ArrowAstropy-like catalog) as a pandas DataFrame."""
    if dataset_name is None:
        return None
    try:
        obj = butler.get(dataset_name)
    except Exception as exc:
        log.warning("Could not load FGCM dataset '%s': %s", dataset_name, exc)
        return None

    # Convert to pandas, going through astropy Table when needed.
    if isinstance(obj, pd.DataFrame):
        df = obj
    elif hasattr(obj, "asAstropy"):
        df = obj.asAstropy().to_pandas()
    elif hasattr(obj, "to_pandas"):
        df = obj.to_pandas()
    else:
        raise TypeError(f"Unsupported FGCM object type for dataset '{dataset_name}': {type(obj)}")

    # Decode byte-string columns (common when the underlying storage is FITS).
    for col in df.columns:
        if df[col].dtype == object:
            sample = df[col].dropna().iloc[0] if df[col].notna().any() else None
            if isinstance(sample, bytes):
                df[col] = df[col].apply(lambda v: v.decode("utf-8") if isinstance(v, bytes) else v)

    log.info("Loaded FGCM dataset '%s': shape=%s", dataset_name, df.shape)
    return df


df_atm_raw = load_fgcm_table(butler, ATM_DATASET)
df_zpt_raw = load_fgcm_table(butler, ZPT_DATASET)

if df_atm_raw is not None:
    print("AtmosphereParameters columns:")
    print(df_atm_raw.columns.tolist())
if df_zpt_raw is not None:
    print("\nZeropoints columns:")
    print(df_zpt_raw.columns.tolist())

## 6. Resolve column names in the FGCM tables

In [ ]:
atm_visit_col = None
atm_param_cols: list[str] = []
if df_atm_raw is not None:
    atm_visit_col = find_col(df_atm_raw, VISIT_COL_CANDIDATES)
    atm_param_cols = [c for c in ATM_PARAM_CANDIDATES if c in df_atm_raw.columns]
    log.info("AtmosphereParameters: visit column = '%s'", atm_visit_col)
    log.info("AtmosphereParameters: atmospheric parameter columns found = %s", atm_param_cols)
    if not atm_param_cols:
        log.warning(
            "None of the expected atmospheric-parameter columns %s were found "
            "in the FGCM AtmosphereParameters table. Inspect df_atm_raw.columns "
            "and update ATM_PARAM_CANDIDATES if needed.",
            ATM_PARAM_CANDIDATES,
        )

zpt_visit_col = None
zpt_det_col = None
zpt_zp_col = None
zpt_zperr_col = None
if df_zpt_raw is not None:
    zpt_visit_col = find_col(df_zpt_raw, VISIT_COL_CANDIDATES)
    try:
        zpt_zp_col = find_col(df_zpt_raw, ZP_COL_CANDIDATES)
    except KeyError:
        log.warning(
            "No FGCM zero-point column found among %s. Inspect df_zpt_raw.columns "
            "and update ZP_COL_CANDIDATES.",
            ZP_COL_CANDIDATES,
        )
    try:
        zpt_zperr_col = find_col(df_zpt_raw, ZP_ERR_COL_CANDIDATES)
    except KeyError:
        log.info("No FGCM zero-point error column found (optional) among %s.", ZP_ERR_COL_CANDIDATES)
    try:
        zpt_det_col = find_col(df_zpt_raw, DET_COL_CANDIDATES)
    except KeyError:
        log.info("No detector column found in FGCM Zeropoints table — will match on visit only.")

    log.info(
        "Zeropoints: visit='%s'  detector='%s'  zp='%s'  zp_err='%s'",
        zpt_visit_col,
        zpt_det_col,
        zpt_zp_col,
        zpt_zperr_col,
    )

## 7. Load the global light-curve table (output of notebook 05)

In [ ]:
lc_path = os.path.join(DIR_LC_IN, GLOBAL_LC_FILE)
log.info("Loading: %s", lc_path)
df_all = pd.read_csv(lc_path)
log.info("Shape: %s  |  columns: %s", df_all.shape, df_all.columns.tolist())
df_all.head(3)

In [ ]:
# Verify required columns coming from notebook 05.
REQUIRED_COLS = ["simbad_id", "visit", "detector_id", "band"]
missing = [c for c in REQUIRED_COLS if c not in df_all.columns]
if missing:
    raise ValueError(f"Required columns missing from input LC file: {missing}")
log.info("All required columns present.")

df_all["visit"] = df_all["visit"].astype(np.int64)
df_all["detector_id"] = df_all["detector_id"].astype(np.int64)

n_rows_before = len(df_all)
log.info("Loaded %d rows for %d unique stars.", n_rows_before, df_all["simbad_id"].nunique())

## 8. Prepare the FGCM sub-tables for merging

In [ ]:
# ── Atmospheric parameters: one row per visit ─────────────────────────────────
df_atm_sub = None
if df_atm_raw is not None and atm_param_cols:
    cols = [atm_visit_col] + atm_param_cols
    df_atm_sub = df_atm_raw[cols].copy()
    rename_map = {atm_visit_col: "visit"}
    rename_map.update({c: f"fgcm_{c.lower()}" for c in atm_param_cols})
    df_atm_sub = df_atm_sub.rename(columns=rename_map)
    df_atm_sub["visit"] = df_atm_sub["visit"].astype(np.int64)
    n_before_dedup = len(df_atm_sub)
    df_atm_sub = df_atm_sub.drop_duplicates(subset=["visit"])
    log.info(
        "Atmospheric parameters sub-table: %d rows (%d before de-duplication on visit).",
        len(df_atm_sub),
        n_before_dedup,
    )
else:
    log.warning("No atmospheric-parameter sub-table built — columns will be skipped.")

# ── FGCM zero-points: one row per (visit[, detector]) ─────────────────────────
df_zpt_sub = None
fgcm_zpt_merge_keys: list[str] = []
if df_zpt_raw is not None and zpt_zp_col is not None:
    cols = [zpt_visit_col, zpt_zp_col]
    rename_map = {zpt_visit_col: "visit", zpt_zp_col: "fgcm_zeropoint"}
    if zpt_zperr_col is not None:
        cols.append(zpt_zperr_col)
        rename_map[zpt_zperr_col] = "fgcm_zeropoint_err"
    if zpt_det_col is not None:
        cols.append(zpt_det_col)
        rename_map[zpt_det_col] = "detector_id"

    df_zpt_sub = df_zpt_raw[cols].copy().rename(columns=rename_map)
    df_zpt_sub["visit"] = df_zpt_sub["visit"].astype(np.int64)

    if "detector_id" in df_zpt_sub.columns:
        df_zpt_sub["detector_id"] = df_zpt_sub["detector_id"].astype(np.int64)
        fgcm_zpt_merge_keys = ["visit", "detector_id"]
    else:
        fgcm_zpt_merge_keys = ["visit"]

    n_before_dedup = len(df_zpt_sub)
    df_zpt_sub = df_zpt_sub.drop_duplicates(subset=fgcm_zpt_merge_keys)
    log.info(
        "FGCM zero-point sub-table: %d rows (%d before de-duplication), merge keys=%s.",
        len(df_zpt_sub),
        n_before_dedup,
        fgcm_zpt_merge_keys,
    )
else:
    log.warning("No FGCM zero-point sub-table built — 'fgcm_zeropoint' column will be skipped.")

## 9. Merge FGCM information into the light-curve table

Both merges are `left` joins: every row of the light-curve table is kept,
and FGCM columns are filled with `NaN` whenever no match is found (missing
FGCM solution for that visit / detector). This is expected and is captured
by the boolean flag columns added at the end of this section.

In [ ]:
# Atmospheric parameters (per visit)
if df_atm_sub is not None:
    df_all = df_all.merge(df_atm_sub, on="visit", how="left")
    fgcm_atm_cols = [c for c in df_atm_sub.columns if c.startswith("fgcm_")]
else:
    fgcm_atm_cols = []

# FGCM zero-point (per visit, or per visit+detector)
if df_zpt_sub is not None:
    df_all = df_all.merge(df_zpt_sub, on=fgcm_zpt_merge_keys, how="left")

log.info("Shape after FGCM merge: %s", df_all.shape)

In [ ]:
# ── Flag columns ───────────────────────────────────────────────────────────────
if fgcm_atm_cols:
    df_all["fgcm_atm_flag"] = df_all[fgcm_atm_cols].notna().any(axis=1)
else:
    df_all["fgcm_atm_flag"] = False

if "fgcm_zeropoint" in df_all.columns:
    df_all["fgcm_zpt_flag"] = df_all["fgcm_zeropoint"].notna()
else:
    df_all["fgcm_zeropoint"] = np.nan
    df_all["fgcm_zeropoint_err"] = np.nan
    df_all["fgcm_zpt_flag"] = False

df_all["fgcm_flag"] = df_all["fgcm_atm_flag"] | df_all["fgcm_zpt_flag"]

log.info(
    "FGCM flags — atm available: %d/%d rows | zpt available: %d/%d rows | any: %d/%d rows",
    int(df_all["fgcm_atm_flag"].sum()),
    len(df_all),
    int(df_all["fgcm_zpt_flag"].sum()),
    len(df_all),
    int(df_all["fgcm_flag"].sum()),
    len(df_all),
)

## 10. Sanity check: row count preserved by the merge

In [ ]:
if len(df_all) != n_rows_before:
    log.warning(
        "Row count changed after merge! before=%d after=%d — check for duplicate "
        "visit/detector entries in the FGCM tables.",
        n_rows_before,
        len(df_all),
    )
else:
    log.info("Row count preserved by the merge (%d rows) — OK.", len(df_all))

## 11. Save the global enriched file

In [ ]:
def save_fgcm(df: pd.DataFrame, out_dir: str, basename: str) -> None:
    """Save enriched DataFrame as both CSV and Parquet."""
    csv_path = os.path.join(out_dir, basename + ".csv")
    parquet_path = os.path.join(out_dir, basename + ".parquet")
    df.to_csv(csv_path, index=False)
    df.to_parquet(parquet_path, index=False)
    log.info("Saved CSV    : %s  (%d rows)", csv_path, len(df))
    log.info("Saved Parquet: %s", parquet_path)

In [ ]:
save_fgcm(df_all, DIR_DATA, "all_stars_lightcurves_fgcm")

# Quick preview of the new columns
preview_cols = ["simbad_id", "visit", "band", "detector_id"]
preview_cols += [c for c in fgcm_atm_cols]
preview_cols += ["fgcm_zeropoint", "fgcm_zeropoint_err", "fgcm_atm_flag", "fgcm_zpt_flag", "fgcm_flag"]
df_all[preview_cols].head(8)

## 12. Generate and save per-star files from the enriched global table

In [ ]:
# Re-read the per-star input filenames to preserve the original naming convention
per_star_files = sorted(f for f in os.listdir(DIR_LC_PER_STAR_IN) if f.endswith("_lc_calib.csv"))
log.info("Found %d per-star CSV files in %s", len(per_star_files), DIR_LC_PER_STAR_IN)

n_ok_ps = 0
n_err_ps = 0

for fname in per_star_files:
    src_path = os.path.join(DIR_LC_PER_STAR_IN, fname)
    try:
        df_star_orig = pd.read_csv(src_path)
        df_star_orig["visit"] = df_star_orig["visit"].astype(np.int64)
        df_star_orig["detector_id"] = df_star_orig["detector_id"].astype(np.int64)
    except Exception as exc:
        log.error("ERROR reading %s: %s", fname, exc)
        n_err_ps += 1
        continue

    # Identify the star(s) in this file
    star_ids_in_file = df_star_orig["simbad_id"].unique()

    # Pull the enriched rows from the global table (already computed above),
    # matching on simbad_id + visit + detector_id to uniquely identify rows.
    df_star_fgcm = df_all.loc[
        df_all["simbad_id"].isin(star_ids_in_file)
        & df_all["visit"].isin(df_star_orig["visit"])
        & df_all["detector_id"].isin(df_star_orig["detector_id"])
    ].copy()

    stem = fname.replace("_lc_calib.csv", "")
    out_stem = stem + "_lc_fgcm"
    save_fgcm(df_star_fgcm, DIR_DATA_PER_STAR, out_stem)
    n_ok_ps += 1

log.info("Per-star files saved: %d OK, %d errors.", n_ok_ps, n_err_ps)

## 13. Diagnostics

In [ ]:
# FGCM coverage per band
coverage_summary = df_all.groupby("band").agg(
    n_rows=("fgcm_flag", "size"),
    n_atm_ok=("fgcm_atm_flag", "sum"),
    n_zpt_ok=("fgcm_zpt_flag", "sum"),
    n_any_ok=("fgcm_flag", "sum"),
)
coverage_summary["frac_atm_ok"] = coverage_summary["n_atm_ok"] / coverage_summary["n_rows"]
coverage_summary["frac_zpt_ok"] = coverage_summary["n_zpt_ok"] / coverage_summary["n_rows"]
print("FGCM coverage per band:")
display(coverage_summary)

In [ ]:
# Number of distinct visits with / without an FGCM zero-point
visits_all = df_all["visit"].unique()
visits_with_zpt = df_all.loc[df_all["fgcm_zpt_flag"], "visit"].unique()
log.info(
    "Distinct visits: %d total | %d with an FGCM zero-point (%.1f%%) | %d without.",
    len(visits_all),
    len(visits_with_zpt),
    100.0 * len(visits_with_zpt) / len(visits_all) if len(visits_all) else 0,
    len(visits_all) - len(visits_with_zpt),
)

## 14. Quick-look plots

In [ ]:
def savefig(fig, name, dpi=150):
    """Save figure as both PDF and PNG."""
    fig.savefig(os.path.join(DIR_FIGS, name + ".pdf"), dpi=dpi, bbox_inches="tight")
    fig.savefig(os.path.join(DIR_FIGS, name + ".png"), dpi=dpi, bbox_inches="tight")
    log.info("Figure saved: %s (.pdf/.png)", os.path.join(DIR_FIGS, name))


bands_available = sorted(df_all["band"].dropna().unique())
n_bands = len(bands_available)
cmap = plt.get_cmap("tab10")

In [ ]:
# ── Plot 1: FGCM zero-point vs PhotoCalib zero-point (comparison) ─────────────
if "zeropoint" in df_all.columns and n_bands > 0 and df_all["fgcm_zpt_flag"].any():
    fig, ax = plt.subplots(figsize=(6, 6))
    for band in bands_available:
        sub = df_all[
            (df_all["band"] == band) & df_all["fgcm_zeropoint"].notna() & df_all["zeropoint"].notna()
        ].drop_duplicates(subset=["visit", "detector_id"])
        if len(sub) == 0:
            continue
        color = cmap(bands_available.index(band))
        ax.scatter(sub["zeropoint"], sub["fgcm_zeropoint"], s=8, alpha=0.5, label=band, color=color)
    lims = df_all[["zeropoint", "fgcm_zeropoint"]].dropna()
    if len(lims) > 0:
        lo, hi = lims.min().min(), lims.max().max()
        ax.plot([lo, hi], [lo, hi], "k--", lw=1, label="1:1")
    ax.set_xlabel("PhotoCalib zero-point  [AB mag]  (notebook 05)")
    ax.set_ylabel("FGCM zero-point  [AB mag]")
    ax.set_title("FGCM vs. PhotoCalib zero-point — one point per (visit, detector)")
    ax.legend(loc="best", fontsize=8)
    plt.tight_layout()
    savefig(fig, "fgcm_vs_photocalib_zeropoint")
    plt.show()
else:
    log.info("Skipping FGCM-vs-PhotoCalib zero-point plot (no data).")

In [ ]:
# ── Plot 2: FGCM zero-point vs expMidptMJD per band ───────────────────────────
if "expMidptMJD" in df_all.columns and n_bands > 0 and df_all["fgcm_zpt_flag"].any():
    fig, ax = plt.subplots(figsize=(12, 5))
    for band in bands_available:
        sub = df_all[(df_all["band"] == band) & df_all["fgcm_zeropoint"].notna()].drop_duplicates(
            subset=["visit", "detector_id"]
        )
        if len(sub) == 0:
            continue
        color = cmap(bands_available.index(band))
        ax.scatter(sub["expMidptMJD"], sub["fgcm_zeropoint"], s=10, alpha=0.6, label=band, color=color)
    ax.set_xlabel("expMidptMJD")
    ax.set_ylabel("FGCM zero-point  [AB mag]")
    ax.set_title("FGCM zero-point vs. time — one point per (visit, detector)")
    ax.legend(loc="best", fontsize=8)
    plt.tight_layout()
    savefig(fig, "fgcm_zeropoint_vs_mjd")
    plt.show()
else:
    log.info("Skipping FGCM zero-point vs. time plot (no data).")

In [ ]:
# ── Plot 3: atmospheric parameters vs expMidptMJD ──────────────────────────────
if "expMidptMJD" in df_all.columns and fgcm_atm_cols:
    n_params = len(fgcm_atm_cols)
    fig, axes = plt.subplots(n_params, 1, figsize=(12, 3.5 * n_params), sharex=True)
    if n_params == 1:
        axes = [axes]
    for ax, col in zip(axes, fgcm_atm_cols):
        for band in bands_available:
            sub = df_all[(df_all["band"] == band) & df_all[col].notna()].drop_duplicates(subset=["visit"])
            if len(sub) == 0:
                continue
            color = cmap(bands_available.index(band))
            ax.scatter(sub["expMidptMJD"], sub[col], s=8, alpha=0.5, label=band, color=color)
        ax.set_ylabel(col)
        ax.legend(loc="best", fontsize=7, ncol=n_bands)
    axes[-1].set_xlabel("expMidptMJD")
    fig.suptitle("FGCM atmospheric parameters vs. time — one point per visit", fontsize=12)
    plt.tight_layout()
    savefig(fig, "fgcm_atm_params_vs_mjd")
    plt.show()
else:
    log.info("Skipping atmospheric-parameter plots (no atmospheric columns available).")

In [ ]:
# ── Plot 4: FGCM coverage fraction per band (bar chart) ────────────────────────
if n_bands > 0:
    fig, ax = plt.subplots(figsize=(6, 4))
    x = np.arange(n_bands)
    width = 0.35
    frac_atm = coverage_summary.loc[bands_available, "frac_atm_ok"].values
    frac_zpt = coverage_summary.loc[bands_available, "frac_zpt_ok"].values
    ax.bar(x - width / 2, 100 * frac_atm, width, label="atmospheric params")
    ax.bar(x + width / 2, 100 * frac_zpt, width, label="zero-point")
    ax.set_xticks(x)
    ax.set_xticklabels(bands_available)
    ax.set_ylabel("FGCM coverage  [%]")
    ax.set_title("Fraction of light-curve points with an FGCM match, per band")
    ax.legend(loc="best", fontsize=8)
    ax.set_ylim(0, 105)
    plt.tight_layout()
    savefig(fig, "fgcm_coverage_fraction_per_band")
    plt.show()

## 15. Output directory listing

In [ ]:
print(f"\n=== Contents of {DIR_DATA} ===")
for entry in sorted(os.listdir(DIR_DATA)):
    full = os.path.join(DIR_DATA, entry)
    if os.path.isdir(full):
        n = len(os.listdir(full))
        print(f"  [DIR]  {entry}/  ({n} files)")
    else:
        size_kb = os.path.getsize(full) / 1024
        print(f"  [FILE] {entry}  ({size_kb:.1f} kB)")